<a href="https://colab.research.google.com/github/21centjoe/NELOS-Quantum-Vector/blob/main/create_a_collab_that_makes_a_an_oscilloscope_It_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Here is a complete, self-contained Google Colab script that builds an interactive, real-time oscilloscope. It captures audio directly from your microphone, visualizes the waveform using HTML5 Canvas, and generates a clean synthesizer tone at the exact same time.

Because Google Colab runs in a cloud environment, standard Python audio libraries (like `pyaudio` or `sounddevice`) cannot access your local hardware. To fix this, the script uses **JavaScript injection** to run the audio processing and rendering directly inside your browser tab via the Web Audio API.

### How to use this in Google Colab:

1. Create a new notebook in Google Colab.
2. Paste the code block below into a single code cell.
3. Run the cell.
4. Click **"Allow Microphone Access"** when prompted by your browser.
5. Use the sliders to change the sound frequency or adjust the oscilloscope's visual zoom.

In [2]:
import IPython
from IPython.display import display, HTML

# JavaScript/HTML template to bridge the browser's hardware with Colab
oscilloscope_html = """
<div style="background-color: #121212; padding: 20px; border-radius: 10px; font-family: monospace; color: #00ffcc; max-width: 640px; margin: auto;">
    <h3 style="margin-top: 0; text-align: center; letter-spacing: 2px;">PYTHIAN AUDIO OSCILLOSCOPE</h3>

    <div style="text-align: center; margin-bottom: 15px;">
        <canvas id="scopeCanvas" width="600" height="250" style="border: 2px solid #00ffcc; background-color: #050505; border-radius: 5px;"></canvas>
    </div>

    <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 15px; background: #1a1a1a; padding: 15px; border-radius: 5px;">
        <div>
            <label style="display: block; margin-bottom: 5px;">EMISSION FREQUENCY: <span id="freqVal">440</span> Hz</label>
            <input type="range" id="frequencySlider" min="100" max="1200" step="1" value="440" style="width: 100%; accent-color: #00ffcc;">

            <label style="display: block; margin-top: 15px; margin-bottom: 5px;">EMISSION VOLUME</label>
            <input type="range" id="volumeSlider" min="0" max="0.5" step="0.01" value="0.1" style="width: 100%; accent-color: #00ffcc;">
        </div>

        <div>
            <label style="display: block; margin-bottom: 5px;">WAVEFORM TYPE</label>
            <select id="waveType" style="width: 100%; background: #222; color: #00ffcc; border: 1px solid #00ffcc; padding: 5px; border-radius: 3px;">
                <option value="sine">Sine (Pure Tone)</option>
                <option value="triangle">Triangle (Soft Harmonics)</option>
                <option value="sawtooth">Sawtooth (Rich Asymmetry)</option>
                <option value="square">Square (Hollow / Digital)</option>
            </select>

            <label style="display: block; margin-top: 15px; margin-bottom: 5px;">VISUAL TIME ZOOM</label>
            <input type="range" id="zoomSlider" min="1" max="5" step="1" value="1" style="width: 100%; accent-color: #00ffcc;">
        </div>
    </div>

    <div style="text-align: center; margin-top: 15px;">
        <button id="startAudioBtn" style="background: #00ffcc; color: #000; border: none; padding: 10px 20px; font-weight: bold; border-radius: 5px; cursor: pointer; letter-spacing: 1px;">
            INITIALIZE AUDIO ARRAY
        </button>
    </div>
</div>

<script>
(async function() {
    const startBtn = document.getElementById('startAudioBtn');
    const canvas = document.getElementById('scopeCanvas');
    const canvasCtx = canvas.getContext('2d');

    const freqSlider = document.getElementById('frequencySlider');
    const freqVal = document.getElementById('freqVal');
    const volSlider = document.getElementById('volumeSlider');
    const waveTypeSelect = document.getElementById('waveType');
    const zoomSlider = document.getElementById('zoomSlider');

    let audioCtx = null;
    let analyser = null;
    let oscillator = null;
    let gainNode = null;
    let stream = null;

    startBtn.onclick = async () => {
        if (audioCtx) return; // Already running

        try {
            // Initialize the Web Audio Context
            audioCtx = new (window.AudioContext || window.webkitAudioContext)();

            // 1. INPUT SETUP: Grab microphone stream
            stream = await navigator.mediaDevices.getUserMedia({ audio: true, video: false });
            const source = audioCtx.createMediaStreamSource(stream);

            // 2. ANALYSER SETUP: The core engine mapping the incoming signal
            analyser = audioCtx.createAnalyser();
            analyser.fftSize = 2048;
            source.connect(analyser);

            // 3. OUTPUT SETUP: Create the sound generator running in parallel
            oscillator = audioCtx.createOscillator();
            gainNode = audioCtx.createGain();

            oscillator.type = waveTypeSelect.value;
            oscillator.frequency.value = parseFloat(freqSlider.value);
            gainNode.gain.value = parseFloat(volSlider.value);

            // Route the generator output back into the analyser so it registers on screen,
            // and connect it to the physical speakers
            oscillator.connect(analyser);
            gainNode.connect(audioCtx.destination);
            oscillator.connect(gainNode);

            oscillator.start();

            startBtn.innerText = "ARRAY SYSTEM ACTIVE";
            startBtn.style.background = "#333";
            startBtn.style.color = "#00ffcc";
            startBtn.disabled = true;

            // Begin the continuous render loop
            drawScope();

        } catch (err) {
            console.error("Audio system failed to hook into hardware:", err);
            alert("Microphone access denied or unsupported browser environment.");
        }
    };

    // Dynamic control update loops
    freqSlider.oninput = () => {
        freqVal.innerText = freqSlider.value;
        if (oscillator) oscillator.frequency.value = parseFloat(freqSlider.value);
    };

    volSlider.oninput = () => {
        if (gainNode) gainNode.gain.value = parseFloat(volSlider.value);
    };

    waveTypeSelect.onchange = () => {
        if (oscillator) oscillator.type = waveTypeSelect.value;
    };

    // Rendering Engine: Translates mathematical data points into structural vectors
    function drawScope() {
        requestAnimationFrame(drawScope);

        const bufferLength = analyser.frequencyBinCount;
        const dataArray = new Uint8Array(bufferLength);

        // Extract time-domain data (the wave's shape)
        analyser.getByteTimeDomainData(dataArray);

        // Clear canvas with a slight trail effect
        canvasCtx.fillStyle = 'rgba(5, 5, 5, 0.3)';
        canvasCtx.fillRect(0, 0, canvas.width, canvas.height);

        // Draw the background grid matrix
        canvasCtx.lineWidth = 1;
        canvasCtx.strokeStyle = 'rgba(0, 255, 204, 0.08)';

        // Vertical lines
        for (let i = 0; i < canvas.width; i += 40) {
            canvasCtx.beginPath();
            canvasCtx.moveTo(i, 0);
            canvasCtx.lineTo(i, canvas.height);
            canvasCtx.stroke();
        }
        // Horizontal lines
        for (let j = 0; j < canvas.height; j += 40) {
            canvasCtx.beginPath();
            canvasCtx.moveTo(0, j);
            canvasCtx.lineTo(canvas.width, j);
            canvasCtx.stroke();
        }

        // Draw the active audio waveform
        canvasCtx.lineWidth = 2.5;
        canvasCtx.strokeStyle = '#00ffcc';
        canvasCtx.shadowBlur = 4;
        canvasCtx.shadowColor = '#00ffcc';
        canvasCtx.beginPath();

        const zoom = parseInt(zoomSlider.value);
        const sliceWidth = (canvas.width * zoom) / bufferLength;
        let x = 0;

        for (let i = 0; i < bufferLength; i++) {
            // Normalize the byte value (0-255) into a fractional float (-1.0 to 1.0)
            const v = dataArray[i] / 128.0;
            const y = (v * canvas.height) / 2;

            if (i === 0) {
                canvasCtx.moveTo(x, y);
            } else {
                canvasCtx.lineTo(x, y);
            }

            x += sliceWidth;
            if (x > canvas.width) break; // Keep rendering inside visible bounds
        }

        canvasCtx.lineTo(canvas.width, canvas.height / 2);
        canvasCtx.stroke();
        canvasCtx.shadowBlur = 0; // Reset shadow for grid loop efficiency
    }
})();
</script>
"""

# Render inside the notebook output cell
display(HTML(oscilloscope_html))